# L1c Example: Float32 Representation and Precision

This companion example repeats the representation calculation for a 32-bit floating-point value and connects the smaller word size to precision and range.

> __Learning Objectives:__
>
> By the end of this example, you should be able to:
> * __Locate and reassemble the fields of a `Float32`:__ Identify the sign bit, the eight exponent bits, and the 23 stored fraction bits in a 32-bit value, then reconstruct the original number from those three parts.
> * __Estimate precision from the significand:__ Derive the machine epsilon from the number of significand bits and convert it into an approximate count of decimal digits you can trust.
> * __Explain what bounds the range:__ Describe how the reserved exponent patterns fix the largest and smallest representable values, and why fewer bits trade both precision and range for storage.

Let's get started!
___

## Setup, Data, and Prerequisites

Run the local setup cell first. It activates the single pinned course environment, loads every package used by this meeting, and includes any meeting source code.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [17]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

LoadError: LoadError: failed to find source of parent package: "FillArrays"
in expression starting at /Users/williammanno/CHEME-5800-CourseRepository-Fall-2026/Include.jl:22
in expression starting at /Users/williammanno/CHEME-5800-CourseRepository-Fall-2026/weeks/week-01/L1c/Include.jl:36

The course environment also loads [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl); see [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/). This notebook does not need it; this notebook uses only Julia's `Base` library. We start using the package later in the course.

___

## Example 32-bit memory layout

<div>
    <center>
        <img src="figs/Fig-Float32-bit-pattern.svg" width="680"/>
    </center>
</div>

Suppose we have a floating point number $x\in\mathbb{R}$ that is approximated as a 32-bit variable in memory. A __finite, normalized__ 32-bit number $x\in\mathbb{R}$ is encoded in memory as:
$$
\begin{align*}
x = \underbrace{S}_{\text{sign}}\times\underbrace{\text{significand}}_{1\,+\,\text{stored fraction}}\times\underbrace{{2^{E-127}}}_{\text{scale}}
\end{align*}
$$
where:
$$
\begin{align*}
S &= (-1)^{d_{31}}\\
\text{significand} &= 1 + \sum_{i = 1}^{23}d_{23-i}2^{-i}\\
E &= \sum_{i=23}^{30}d_{i}2^{i - 23}
\end{align*}
$$
where $d_{i}$ denotes the digit at position $i$ in the number.

> __Watch the indexing in the significand.__ The digit index counts _down_ while the weight index counts _up_: $d_{22}$ (just below the exponent field) carries $2^{-1}$, $d_{21}$ carries $2^{-2}$, and so on to $d_{0}$ carrying $2^{-23}$. That is what the $d_{23-i}$ subscript encodes, and our implementation uses the same indexing.

Notice the difference between the 64- and 32-bit numbers: the number of elements used to compute the significand and the exponent terms are different, and the location of the sign bit has changed, but otherwise they have a similar structural layout in memory.

Now, let's compute the components of the 32-bit representation of $x\in\mathbb{R}$. First, specify an example number, save it in the `x::Float32` variable:

In [2]:
x = 141.72 |> Float32; # why do we need |> Float32?

> __Why do we need the `|> Float32`?__ An unsuffixed literal like `141.72` is a `Float64` in Julia; that was the default we saw in `L1a`. Without the explicit conversion, [the `bitstring(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.bitstring) would hand us 64 bits and none of the 32-bit indices would line up. (The alternative spelling is the literal suffix: `141.72f0`.)

Check the type using [the `typeof(...)` function](https://docs.julialang.org/en/v1/base/base/#Core.typeof):

In [3]:
typeof(x) == Float32 # if Float32, this should be true

true

Next, let's use [the `bitstring(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.bitstring) to generate the bits of our 32-bit floating point number $x$ as a `String`, and then convert and save the bitstring into a `0`-based dictionary called `d::Dict{Int,Int}`:

In [4]:
d = let

    # initialize -
    bitpattern_dictionary = Dict{Int64,Int64}(); # storage for the 0-based bit pattern
    wordsize = 32; # how many boxes do we have?
    a = bitstring(x) |> reverse |> collect .|> v-> parse(Int64, v) # fancy. Nothing to see here, move along (for now anyway).
    
    # put stuff in the dictionary
    for i ∈ 0:(wordsize-1)
        bitpattern_dictionary[i] = a[i+1];
    end
    bitpattern_dictionary # return to caller
end;

### Sign term
Now that we have the bitpattern dictionary `d::Dict{Int, Int}`, we can compute the three components of our floating point number. Let's start with the sign. The `S::Float64` variable holds either `+1.0` or `-1.0`, depending on the single bit at position 31:

In [5]:
S = let
    S = (-1.0)^(d[31]);
end

1.0

### Significand
Next, let's compute a value for the [significand](https://docs.julialang.org/en/v1/base/numbers/#Base.Math.significand) of $x\in\mathbb{R}$, which we'll store in the `calculated_significand_value::Float64` variable:

In [6]:
calculated_significand_value = let

    # initialize -
    calculated_significand_value = 0.0;
    b = 2.0; # binary, base = 2
    number_of_fraction_bits = 23; # the stored fraction is d[22] down to d[0]
    significand_range_array = range(1,stop=number_of_fraction_bits,step=1) |> collect; # the weights 2^-1 ... 2^-23

    for i ∈ significand_range_array
        calculated_significand_value += (b^(-i))*d[number_of_fraction_bits-i]
    end
    calculated_significand_value + 1 # don't forget to add 1!
end

1.1071875095367432

__Check__: Let's use [the `@assert` macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert) to check our calculated significand value against the output of [the `significand(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.Math.significand) using [the `==` comparison operator](https://docs.julialang.org/en/v1/manual/missing/#Equality-and-Comparison-Operators). 
> _What happens_? If [the `==` comparison](https://docs.julialang.org/en/v1/manual/missing/#Equality-and-Comparison-Operators) comes back `false`, [an `AssertionError` is thrown](https://docs.julialang.org/en/v1/base/base/#Core.AssertionError) (and we know something is wrong with our calculation). However, if the comparison comes back `true`, we can be confident that our calculation is correct (no error is thrown).

So what happens?

In [7]:
@assert significand(x) == calculated_significand_value # compare built-in versus our calculated value

### Scale term
Now, let's compute the scale of the floating point number $x\in\mathbb{R}$, which requires us to calculate the exponent value $E$, which we'll store in the `E::Float64` variable. 
    
_Aside_: Sometimes you'll see the exponent $E$ expression for a 32-bit floating point number written as:
$$
E = \sum_{i=0}^{7}e_{i}2^{i}
$$
where the $e_{i}$'s denote _exponent bits_, i.e., digits from the original bit string whose indexes have been remapped to be $0\rightarrow{7}$. In this convention, $e_{0} = d_{23},e_{1} = d_{24},\dots,e_{7} = d_{30}$.  Let's implement the $0\rightarrow{7}$ summation:

In [8]:
E = let

    # initialize -
    calculated_exponent_value = 0.0;
    b = 2.0; # binary, base = 2
    msb = 30; # most significant bit (msb)
    lsb = 23; # least significant bit (lsb)
    exponent_bit_range_array = range(lsb, stop=msb, step = 1) |> collect

    for i ∈ eachindex(exponent_bit_range_array)
        j = exponent_bit_range_array[i]; # remap operation: i runs from 1->8 (notice not zero based), while j runs from lsb -> msb
        calculated_exponent_value += d[j]*(b^(i - 1)) # why -1?
    end
    calculated_exponent_value # return
end;

#### Do we get the same number $x$?
Let's put all the pieces together and check our work. If our calculations are correct, our calculated number should be the same (evaluated [using the `==` comparison operator](https://docs.julialang.org/en/v1/manual/missing/#Equality-and-Comparison-Operators)) as the `x::Float32` value specified above. Let's use [the `@assert` macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert) to check our calculated $x$ value against the original value using [the `==` comparison operator](https://docs.julialang.org/en/v1/manual/missing/#Equality-and-Comparison-Operators). 
> _What happens_? If [the `==` comparison](https://docs.julialang.org/en/v1/manual/missing/#Equality-and-Comparison-Operators) comes back `false`, [an `AssertionError` is thrown](https://docs.julialang.org/en/v1/base/base/#Core.AssertionError), and we know something is wrong with our calculation.

Let's put the pieces together:

In [9]:
@assert S*(calculated_significand_value)*2^(E - 127) == x # If this doesn't blow up, nice!

The assertion passed, but it quietly hid the most interesting number in this notebook. Our reconstruction is a `Float64`, while `x` is a `Float32`. Let's look at what we actually got:

In [10]:
let
    reconstructed = S*(calculated_significand_value)*2^(E - 127) # this is a Float64

    (reconstructed    = reconstructed, # what 141.72 actually became in 32 bits
     literal_as_f64   = 141.72,        # what we asked for
     local_spacing    = eps(x),        # gap between neighbouring Float32 values near x
     equal            = reconstructed == x,  # true:  == promotes the Float32 to Float64
     identical        = reconstructed === x) # false: they are different types
end

(reconstructed = 141.72000122070312, literal_as_f64 = 141.72, local_spacing = 1.5258789f-5, equal = true, identical = false)

There it is: we asked for `141.72` and the machine stored `141.72000122070312`. That gap is not a display artifact and not a bug; it is the nearest value a `Float32` can represent, and it sits within the local spacing reported by [the `eps(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.eps-Tuple%7BAbstractFloat%7D) above. This is the concrete face of the roughly 7 decimal digits we derive in the precision section.

Note also that `==` returned `true` while `===` returned `false`. [The `@assert` macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert) passed because `==` promotes the `Float32` to `Float64` before comparing. Comparison in Julia is doing more work than it appears to, which is worth remembering the next time a test passes and you are not sure why.

___

## Deep dive: How big (small) can the significand be?
The fractional component of the floating-point number is contained in the significand. Thus, an interesting question is how big (or small) can this component be?
* __Idea__: To explore this question, examine the summation term in the significand expression. If all the digits in the summation expression $\left\{d_{1},d_{2},\dots,d_{23}\right\}$ were `0`, then the _smallest possible value_ of the `significand = 1.` Alternatively, if all the digits $\left\{d_{1},d_{2},\dots,d_{23}\right\}$ were `1`, then we'd get a maximum value. What is the maximum possible value?

Let's explore this numerically first. The `max_significand_value::Float64` variable holds the significand we get when every one of the 23 fraction bits is set to `1`.

In [11]:
max_significand_value = let

    # initialize -
    d = Dict{Int64, Int64}(); 
    calculated_significand_value = 0.0;
    b = 2.0; # binary, base = 2
    number_of_fraction_bits = 23; # the stored fraction is d[22] down to d[0]
    significand_range_array = range(1,stop=number_of_fraction_bits,step=1) |> collect; # the weights 2^-1 ... 2^-23

    # all ones, gives max 
    number_of_digits = length(significand_range_array); # how many digits do we have for the significand?
    for i ∈ 1:number_of_digits
        d[i-1] = 1.0;
    end

    for i ∈ significand_range_array
        calculated_significand_value += (b^(-i))*d[number_of_fraction_bits-i]
    end
    calculated_significand_value + 1
end

1.9999998807907104

### Analytical analysis

The numerical calculation gave a `max_significand_value ≈ 2`, i.e., the value of the summation term, is $\approx{1}$. We'd expect this because the summation term is an infinite series in the powers $2^{-i}$ truncated at the number of bits used for the fraction. To see this, let's do a little math. 
$$
\begin{align*}
S_{N} & = \sum_{i=1}^{N}2^{-i} = 2^{-1} + 2^{-2}+\dots+2^{-N}\quad\text{this gives}\,{a = 2^{-1}\,\text{and}\,{r} = 2^{-1}}\\
S_{N} &= \frac{a\left(1-r^{N}\right)}{1-r} = 1-2^{-N}\quad\text{substitute}\,a\,\text{and}\,{r}\,\text{simplify}\\
S_{N} &= 1 - 2^{-23}\approx{0.9999998807907104}\quad{N = 23}\,\blacksquare
\end{align*}
$$
As $N\rightarrow\infty$ the partial sum $S_{N}\rightarrow{1}$. However, we don't get exactly `1` numerically. Why? Because we truncate the series early, i.e, for a 32-bit number we run the series up to $N = 23$, which gives a value slightly less than `1.`

### Precision

The analysis above also gives us insight into the _precision_ of a Float32 value, i.e., the number of possible decimal digits. The precision of a Float32 value is set by its $N = 23$ explicit fraction bits plus one implicit leading bit, so $p=24$. Then, the machine epsilon - the gap between 1.0 and the next representable float - is given by:
$$
\begin{align*}
\epsilon &= 2^{(1-p)}\quad\text{substitute}\,{p=24}\,\,\text{for a 32-bit number}\\
\epsilon & \approx 1.19209\times{10}^{-7}
\end{align*}
$$
This corresponds to $d\approx{-\log_{10}\epsilon}$ digits of precision, which for a `Float32` is $\approx{7}$ digits.

In [12]:
let
    p = 24; # p = {24,53} for {32,64}-bit
    ϵ = 2.0^(1-p);
    d = -log10(ϵ) |> round
end

7.0

___

## How big (small) can the scale be?
Next, let's think about the possible scale of a `Float32` given by: $\text{scale} = 2^{E - 127}$. To explore this question (in a first approximation where we ignore edge cases associated with representing $\pm\infty$ or NaNs), let's start by looking at the possible limits for the $E$ term in the memory layout for $x\in\mathbb{R}$ approximated as a `Float32`.
* __Idea__: The $E$ expression is computed from the summation of 8 bits (the exponent bits). If all the exponent digits $\left\{e_{0},e_{1},\dots,e_{7}\right\}$ were `0`, then the value of the $\text{scale} = 2^{-127}\approx{0}$. Alternatively, if all the exponent digits $\left\{e_{0},e_{1},\dots,e_{7}\right\}$ were `1`, then we'd get a maximum value for $E$. What is the maximum possible value?

Let's compute the maximum permissible value for $E$ numerically, and then think about what we should expect to see analytically. Store the maximum possible $E$ in the `max_possible_E::Float64` variable:

In [13]:
max_possible_E = let

    # initialize -
    d = Dict{Int64, Int64}();
    calculated_exponent_value = 0.0;
    b = 2.0; # binary, base = 2
    msb = 30; # most significant bit (msb)
    lsb = 23; # least significant bit (lsb)
    exponent_bit_range_array = range(lsb, stop=msb, step = 1) |> collect

    # all ones, gives max 
    number_of_digits = length(exponent_bit_range_array); # how many digits do we have for the exponent E?
    for i ∈ 1:number_of_digits
        d[i-1] = 1.0;
    end

    for i ∈ eachindex(exponent_bit_range_array)
        calculated_exponent_value += d[i-1]*(b^(i - 1)) # why -1?
    end
    calculated_exponent_value
end

255.0

We get `255`; however, in practice, our logic has a flaw (edge cases mentioned above)!
* The `255` case is a special reserved case. An exponent bit sequence of all 1's (255) is a special code: if the fraction bits (the bit sequence in the significand calculation) are zero, it means we are representing $\pm\infty$, and if the fraction is nonzero, it means not a number (NaN); hence the maximum finite exponent for non-edge case numbers is `254.`

So `max_possible_E` stays at `255`, and `max_possible_E_usable` holds the corrected value. With `max_possible_E_usable = 254`, the maximum permissible scale will be: $2^{127}$!

In [14]:
max_possible_E_usable = max_possible_E - 1 # 255 is reserved for Inf/NaN, so 254 is the largest usable E
max_scale = 2^(max_possible_E_usable - 127) # wow!

1.7014118346046923e38

__Should we have expected 255__? 

Yes, we should have! Let's look at the exponent summation, assuming all eight exponent bits are `1`, then we have (where $N$ is the index of the most significant bit):
$$
\begin{align*}
E &= \sum_{i=0}^{N}2^{i} = 1+2^{1}+2^{2}+\dots+2^{N-1}+2^{N}\quad\text{multiply by 2}\\
2E &= 2 + 2^{2}+\dots+2^{N}+2^{N+1}\quad\text{subtract}\,2E-E\\
E &= 2^{N+1} - 1\quad\text{substitute}\,{N = 7}\\
E &= 2^{8} - 1 = 255\quad\blacksquare
\end{align*}
$$

Putting the pieces together gives a back-of-the-envelope range for `Float32` of roughly $\pm\left[2^{-127},2^{127}\right]$ with about `7` decimal digits. That estimate is in the right neighbourhood, but __both ends are wrong__, in ways worth understanding.

### The reserved exponent patterns

We have now bumped into the edges twice: once at $E = 255$, once at $E = 0$. These are not special-case clutter bolted onto the format; they are how IEEE-754 encodes everything that is not an ordinary number. Both ends of the exponent field are reserved:

| Exponent $E$ | Fraction bits | Meaning | Value |
|---|---|---|---|
| $0$ | all zero | signed zero | $\pm 0$ |
| $0$ | nonzero | __subnormal__: implicit leading digit is `0`, not `1` | $\pm\,\text{fraction}\times 2^{-126}$ |
| $1\dots254$ | any | __normal__: everything we computed above | $\pm\,(1+\text{fraction})\times 2^{E-127}$ |
| $255$ | all zero | infinity | $\pm\infty$ |
| $255$ | nonzero | not a number | `NaN` |

So the honest numbers are:

* __Smallest normal value:__ the smallest _usable_ exponent is $E = 1$, not $E = 0$, giving $2^{-126}\approx 1.1755\times 10^{-38}$. Our $2^{-127}$ was one power of two too small.
* __Smallest positive value of any kind:__ $2^{-149}\approx 1.4\times 10^{-45}$, the smallest [subnormal](https://en.wikipedia.org/wiki/Subnormal_number). Subnormals give up precision to fill the gap between $2^{-126}$ and zero, which is why the two bounds differ.
* __Largest finite value:__ __not__ $2^{127}$. That is the largest _scale_; the significand contributes up to $2-2^{-23}$ on top of it, giving $(2-2^{-23})\times 2^{127}\approx 3.4028\times 10^{38}$, just under $2^{128}$ and roughly double our estimate.

Let's check all three against Julia's built-ins, [the `floatmin(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.floatmin) and [the `floatmax(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.floatmax):

In [15]:
(smallest_normal    = floatmin(Float32),   ours_normal    = Float32(2.0^-126),
 smallest_subnormal = nextfloat(0.0f0),    ours_subnormal = Float32(2.0^-149),
 largest_finite     = floatmax(Float32),   ours_largest   = Float32((2 - 2.0^-23)*2.0^127))

(smallest_normal = 1.1754944f-38, ours_normal = 1.1754944f-38, smallest_subnormal = 1.0f-45, ours_subnormal = 1.0f-45, largest_finite = 3.4028235f38, ours_largest = 3.4028235f38)

__And the formula only covers the normal case.__ Everything we derived assumed the implicit leading `1`, so it silently returns nonsense for the reserved patterns rather than failing loudly. Watch it break on `0.0f0` and `Inf32`:

In [16]:
reconstruct = function(v::Float32) # exactly the procedure we built above, packaged up
    bits = bitstring(v) |> reverse |> collect .|> c -> parse(Int64, c)
    dd = Dict(i => bits[i+1] for i ∈ 0:31)
    S = (-1.0)^dd[31]
    significand = 1 + sum(2.0^(-i)*dd[23-i] for i ∈ 1:23)
    E = sum(dd[j]*2.0^(j-23) for j ∈ 23:30)
    S*significand*2.0^(E - 127)
end

(zero_in = 0.0f0, zero_out = reconstruct(0.0f0),  # should be 0.0
 inf_in  = Inf32, inf_out  = reconstruct(Inf32))  # should be Inf

(zero_in = 0.0f0, zero_out = 5.877471754111438e-39, inf_in = Inf32, inf_out = 3.402823669209385e38)

Zero comes back as $2^{-127}\approx 5.88\times10^{-39}$, and infinity comes back as a large but finite number. Both failures have the same cause: our formula added an implicit leading `1` that the format never intended for these patterns. This is why real implementations branch on the exponent field _before_ doing any arithmetic, and why the qualifier "finite, normalized" at the top of this notebook was not boilerplate.

___

## Summary
A `Float32` spends its 32 bits on one sign bit, eight exponent bits, and 23 stored fraction bits, and every limit of the format follows from that division.

> __Key Takeaways:__
>
> * **The field widths set the precision:** Twenty-three stored fraction bits plus one implicit leading bit give 24 bits of significand, which works out to roughly seven decimal digits you can rely on.
> * **The reserved exponents set the range:** Because the all-zero and all-ones exponent patterns are reserved for zero, subnormals, infinity, and `NaN`, the raw formula's powers of two are wrong at both ends: the largest finite value is `floatmax(Float32)`, and the bottom has two answers, the smallest normal `floatmin(Float32)` and the far smaller subnormal beneath it.
> * **Choosing a float type is an engineering decision:** Halving the width halves the storage and costs both precision and range, which is a tradeoff to make deliberately rather than by accepting a default.

The same three fields appear in a `Float64`, only wider. Once you can take one format apart, the others follow.
___